# Test DuckDB

## Build
skip for duckdb

## Evaluate

### Init

In [ ]:
import sys
import os
import subprocess
import time
import tempfile
from pathlib import Path

# 添加 experiment 目录和 scripts 目录到路径
sys.path.append(os.path.abspath('.'))
sys.path.append(os.path.abspath('../scripts'))

from ExperimentRunner import DuckDBTestRunner
from extract_card_from_explain import process_data

# 项目根目录
project_root = os.path.abspath('..')

# 输出到 experiment/checkpoint/DuckDB 目录
checkpoint_dir = Path('./checkpoint/DuckDB').resolve()
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# 创建 runner
runner = DuckDBTestRunner(project_root)

# DuckDB 可执行文件路径（在 running_space 目录中）
duckdb_exec = runner.running_space / "duckdb"

def parse_duckdb_explain_output(output_text):
    """
    解析 DuckDB EXPLAIN 输出，提取每个查询的基数估计

    使用 scripts/extract_card_from_explain.py 的 process_data 函数进行解析

    参数:
        output_text: DuckDB EXPLAIN 的输出文本

    返回:
        list: 包含每个查询基数估计的列表（int）

    异常:
        RuntimeError: 解析失败时抛出，包含原始错误信息
    """
    # 使用 extract_card_from_explain.py 的 process_data 函数
    # 该函数使用正则表达式匹配 "数字 Rows" 模式，并以 \n\n 分割数据
    try:
        cardinalities_str = process_data(output_text)
        # 将字符串转换为整数
        cardinalities = [int(card) for card in cardinalities_str]
        return cardinalities
    except Exception as e:
        raise RuntimeError(
            f"DuckDB EXPLAIN 输出解析失败: {e}\n"
            f"可能原因: DuckDB 版本升级改变了 EXPLAIN 输出格式。\n"
            f"请检查 scripts/extract_card_from_explain.py 中的正则表达式是否需要更新。"
        ) from e

def run_duckdb_explain(db_path, explain_sql_file, output_file=None):
    """
    运行 DuckDB EXPLAIN 查询

    参数:
        db_path: 数据库文件路径
        explain_sql_file: 包含 EXPLAIN 查询的 SQL 文件路径
        output_file: 输出文件路径（可选，如果为None则返回输出文本）

    返回:
        str: DuckDB 的输出文本
    """
    if not duckdb_exec.exists():
        raise FileNotFoundError(f"DuckDB 可执行文件不存在: {duckdb_exec}")

    if not Path(db_path).exists():
        raise FileNotFoundError(f"数据库文件不存在: {db_path}")

    if not Path(explain_sql_file).exists():
        raise FileNotFoundError(f"SQL 文件不存在: {explain_sql_file}")

    # 使用输入重定向的方式执行 DuckDB
    # 命令格式：duckdb db_path < explain_sql_file
    try:
        with open(explain_sql_file, 'r', encoding='utf-8') as f:
            sql_content = f.read()

        # 使用 subprocess 执行，通过 stdin 传递 SQL
        result = subprocess.run(
            [str(duckdb_exec), str(db_path)],
            input=sql_content,
            cwd=str(runner.running_space),
            capture_output=True,
            text=True,
            timeout=3600  # 1小时超时
        )

        output_text = result.stdout + result.stderr

        if output_file:
            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(output_text)

        if result.returncode != 0:
            raise RuntimeError(f"DuckDB 执行失败，返回码: {result.returncode}\n{result.stderr}")

        return output_text
    except subprocess.TimeoutExpired:
        raise RuntimeError("DuckDB 执行超时")
    except Exception as e:
        raise RuntimeError(f"运行 DuckDB 时发生错误: {e}")

def evaluate_duckdb_benchmark(benchmark, include_cardinalities: bool = False, preview_size: int = 5):
    """
    评估 DuckDB 对 SQL 查询的基数估计

    参数:
        benchmark: benchmark名称 ('Stats', 'JOBM', 'JOBLight', 'JOBLightRanges', 'JobJoin', 'StatsJoin')
        include_cardinalities: 是否在返回结果中包含完整基数列表
        preview_size: 不返回完整基数时，预览前 N 个基数

    返回:
        dict: 包含评估结果的字典，包括：
            - total_time: 总评估时间（秒）
            - num_queries: 查询数量
            - output_file: 结果输出文件路径
            - cardinalities: 每个查询的基数估计列表（可选）
            - cardinality_preview: 基数预览（可选）
    """
    if benchmark == 'Stats':
        config = runner.get_stats_config()
        queries_file = Path(config.SUBQUERY_PATH)
        db_path = Path(config.DB_PATH)
        result_path = checkpoint_dir / "card_stats.txt"
    elif benchmark == 'JOBM':
        config = runner.get_jobm_config()
        queries_file = Path(config.SUBQUERY_PATH)
        db_path = Path(config.DB_PATH)
        result_path = checkpoint_dir / "card_jobm.txt"
    elif benchmark == 'JOBLight':
        config = runner.get_joblight_config()
        queries_file = Path(config.SUBQUERY_PATH)
        db_path = Path(config.DB_PATH)
        result_path = checkpoint_dir / "card_joblight.txt"
    elif benchmark == 'JOBLightRanges':
        config = runner.get_joblight_ranges_config()
        queries_file = Path(config.SUBQUERY_PATH)
        db_path = Path(config.DB_PATH)
        result_path = checkpoint_dir / "card_joblr.txt"
    elif benchmark == 'JobJoin':
        config = runner.get_jobjoin_config()
        queries_file = Path(config.SQL_PATH)
        db_path = Path(config.DB_PATH)
        result_path = checkpoint_dir / "card_jobjoin.txt"
    elif benchmark == 'StatsJoin':
        config = runner.get_statsjoin_config()
        queries_file = Path(config.SUBQUERY_PATH)
        db_path = Path(config.DB_PATH)
        result_path = checkpoint_dir / "card_statsjoin.txt"
    else:
        raise ValueError(f"不支持的benchmark: {benchmark}")

    print(f"\n{'='*60}")
    print(f"处理 Benchmark: {benchmark}")
    print(f"查询文件: {queries_file}")
    print(f"数据库文件: {db_path}")
    print(f"结果输出: {result_path}")

    # 1. 生成 explain.sql 文件
    explain_sql = runner.running_space / "explain.sql"
    runner._prepare_input_sql_with_explain(queries_file, explain_sql)
    print(f"已生成 EXPLAIN SQL 文件: {explain_sql}")

    # 2. 运行 DuckDB EXPLAIN
    print(f"\n开始运行 DuckDB EXPLAIN...")
    start_time = time.time()

    temp_file = tempfile.NamedTemporaryFile(
        dir=str(runner.running_space),
        suffix=".txt",
        delete=False
    )
    explain_output_path = Path(temp_file.name)
    temp_file.close()

    try:
        output_text = run_duckdb_explain(
            db_path=db_path,
            explain_sql_file=explain_sql,
            output_file=str(explain_output_path)
        )

        total_time = time.time() - start_time
        print(f"DuckDB EXPLAIN 运行完成，耗时: {total_time:.2f} 秒")

        # 3. 解析 EXPLAIN 输出，提取基数估计
        print(f"\n开始解析 EXPLAIN 输出...")
        if explain_output_path.exists():
            with open(explain_output_path, 'r', encoding='utf-8') as f:
                output_text_from_file = f.read()
            cardinalities = parse_duckdb_explain_output(output_text_from_file)
        else:
            cardinalities = parse_duckdb_explain_output(output_text)
        num_queries = len(cardinalities)
        print(f"解析完成，找到 {num_queries} 个查询的基数估计")
    finally:
        # 清理临时 explain 输出文件
        if explain_output_path.exists():
            explain_output_path.unlink()

    # 4. 保存结果到文件（每行一个基数）
    with open(result_path, 'w', encoding='utf-8') as f:
        for card in cardinalities:
            f.write(f"{card}\n")

    print(f"结果已保存到: {result_path}")

    result = {
        'benchmark': benchmark,
        'total_time': total_time,
        'num_queries': num_queries,
        'output_file': str(result_path)
    }

    if include_cardinalities:
        result['cardinalities'] = cardinalities
    else:
        result['cardinality_preview'] = cardinalities[:max(preview_size, 0)]
        result['cardinality_preview_size'] = max(preview_size, 0)

    return result

print("评估函数已定义，可以使用 evaluate_duckdb_benchmark() 函数评估各个 benchmark")

### Stats Benchmark

In [2]:
# 执行评估
stats_results = evaluate_duckdb_benchmark(benchmark='Stats')
stats_results


处理 Benchmark: Stats
查询文件: /home/liwei/starCE/Benchmark/workloads/STATS-CEB/subquery/subquery.sql
数据库文件: /home/liwei/starCE/Benchmark/duckdb/stats.db
结果输出: /home/liwei/starCE/experiment/checkpoint/DuckDB/card_stats.txt
[2026-06-21 13:18:12] 已复制queries文件到 /home/liwei/starCE/experiment/running_space/explain.sql 并添加EXPLAIN
已生成 EXPLAIN SQL 文件: /home/liwei/starCE/experiment/running_space/explain.sql

开始运行 DuckDB EXPLAIN...
DuckDB EXPLAIN 运行完成，耗时: 4.11 秒

开始解析 EXPLAIN 输出...
解析完成，找到 2471 个查询的基数估计
结果已保存到: /home/liwei/starCE/experiment/checkpoint/DuckDB/card_stats.txt


{'benchmark': 'Stats',
 'total_time': 4.1143951416015625,
 'num_queries': 2471,
 'output_file': '/home/liwei/starCE/experiment/checkpoint/DuckDB/card_stats.txt',
 'cardinality_preview': [40240, 6707, 6707, 6707, 201203],
 'cardinality_preview_size': 5}

### JOBM Benchmark

In [3]:
# 执行评估
jobm_results = evaluate_duckdb_benchmark(benchmark='JOBM')
jobm_results


处理 Benchmark: JOBM
查询文件: /home/liwei/starCE/Benchmark/workloads/JOBM/subquery/subquery.sql
数据库文件: /home/liwei/starCE/Benchmark/duckdb/imdb.db
结果输出: /home/liwei/starCE/experiment/checkpoint/DuckDB/card_jobm.txt
[2026-06-21 13:18:17] 已复制queries文件到 /home/liwei/starCE/experiment/running_space/explain.sql 并添加EXPLAIN
已生成 EXPLAIN SQL 文件: /home/liwei/starCE/experiment/running_space/explain.sql

开始运行 DuckDB EXPLAIN...
DuckDB EXPLAIN 运行完成，耗时: 18.64 秒

开始解析 EXPLAIN 输出...
解析完成，找到 6424 个查询的基数估计
结果已保存到: /home/liwei/starCE/experiment/checkpoint/DuckDB/card_jobm.txt


{'benchmark': 'JOBM',
 'total_time': 18.637667179107666,
 'num_queries': 6424,
 'output_file': '/home/liwei/starCE/experiment/checkpoint/DuckDB/card_jobm.txt',
 'cardinality_preview': [0, 300, 7521, 7521, 0],
 'cardinality_preview_size': 5}

### JOBLight Benchmark

In [4]:
# 执行评估
joblight_results = evaluate_duckdb_benchmark(benchmark='JOBLight')
joblight_results


处理 Benchmark: JOBLight
查询文件: /home/liwei/starCE/Benchmark/workloads/JOBLight/subquery/subquery.sql
数据库文件: /home/liwei/starCE/Benchmark/duckdb/imdb.db
结果输出: /home/liwei/starCE/experiment/checkpoint/DuckDB/card_joblight.txt
[2026-06-21 13:18:36] 已复制queries文件到 /home/liwei/starCE/experiment/running_space/explain.sql 并添加EXPLAIN
已生成 EXPLAIN SQL 文件: /home/liwei/starCE/experiment/running_space/explain.sql

开始运行 DuckDB EXPLAIN...
DuckDB EXPLAIN 运行完成，耗时: 0.49 秒

开始解析 EXPLAIN 输出...
解析完成，找到 451 个查询的基数估计
结果已保存到: /home/liwei/starCE/experiment/checkpoint/DuckDB/card_joblight.txt


{'benchmark': 'JOBLight',
 'total_time': 0.48886680603027344,
 'num_queries': 451,
 'output_file': '/home/liwei/starCE/experiment/checkpoint/DuckDB/card_joblight.txt',
 'cardinality_preview': [3912808, 3912808, 3912808, 3912808, 46953704],
 'cardinality_preview_size': 5}

### JOBLightRanges Benchmark

In [5]:
# 执行评估
joblr_results = evaluate_duckdb_benchmark(benchmark='JOBLightRanges')
joblr_results


处理 Benchmark: JOBLightRanges
查询文件: /home/liwei/starCE/Benchmark/workloads/JOBLightRanges/subquery/subquery.sql
数据库文件: /home/liwei/starCE/Benchmark/duckdb/imdb.db
结果输出: /home/liwei/starCE/experiment/checkpoint/DuckDB/card_joblr.txt
[2026-06-21 13:18:36] 已复制queries文件到 /home/liwei/starCE/experiment/running_space/explain.sql 并添加EXPLAIN
已生成 EXPLAIN SQL 文件: /home/liwei/starCE/experiment/running_space/explain.sql

开始运行 DuckDB EXPLAIN...
DuckDB EXPLAIN 运行完成，耗时: 8.87 秒

开始解析 EXPLAIN 输出...
解析完成，找到 8292 个查询的基数估计
结果已保存到: /home/liwei/starCE/experiment/checkpoint/DuckDB/card_joblr.txt


{'benchmark': 'JOBLightRanges',
 'total_time': 8.867777347564697,
 'num_queries': 8292,
 'output_file': '/home/liwei/starCE/experiment/checkpoint/DuckDB/card_joblr.txt',
 'cardinality_preview': [1956405, 4695371, 1956405, 3912808, 1956405],
 'cardinality_preview_size': 5}

### JobJoin Benchmark

In [6]:
# 执行评估
jobjoin_results = evaluate_duckdb_benchmark(benchmark='JobJoin')
jobjoin_results


处理 Benchmark: JobJoin
查询文件: /home/liwei/project/starCE/Benchmark/workloads/JobJoin/subquery/subquery.sql
数据库文件: /home/liwei/project/starCE/Benchmark/duckdb/imdb.db
结果输出: /home/liwei/project/starCE/experiment/checkpoint/DuckDB/card_jobjoin.txt
[2026-06-21 21:45:58] 已复制queries文件到 /home/liwei/project/starCE/experiment/running_space/explain.sql 并添加EXPLAIN
已生成 EXPLAIN SQL 文件: /home/liwei/project/starCE/experiment/running_space/explain.sql

开始运行 DuckDB EXPLAIN...
DuckDB EXPLAIN 运行完成，耗时: 12.77 秒

开始解析 EXPLAIN 输出...
解析完成，找到 9565 个查询的基数估计
结果已保存到: /home/liwei/project/starCE/experiment/checkpoint/DuckDB/card_jobjoin.txt


{'benchmark': 'JobJoin',
 'total_time': 12.772614240646362,
 'num_queries': 9565,
 'output_file': '/home/liwei/project/starCE/experiment/checkpoint/DuckDB/card_jobjoin.txt',
 'cardinality_preview': [2609129, 3378693, 3590825, 1787077, 1380035],
 'cardinality_preview_size': 5}

### StatsJoin Benchmark

In [ ]:
# 执行评估
statsjoin_results = evaluate_duckdb_benchmark(benchmark='StatsJoin')
statsjoin_results